# 02 - Feature engineering

What `build_features` produces, and the checks that matter.

> **These notebooks define no functions.** Everything they call lives in `src/`.
> That rule is from `brain.md` section 7: logic written in a cell cannot be tested
> and silently drifts from the module, which is how report figures stop matching
> the code that ships.
>
> Until real case data is in `data/raw/cases/`, these fall back to a generated
> panel and every number describes a generator rather than dengue.

In [ ]:
import sys
import warnings

sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

from src.config import load_config

cfg = load_config("../config.yaml")
cfg.project.name, cfg.project.granularity

In [ ]:
from src.panel import assemble_panel
from src.preprocess import preprocess

try:
    panel = assemble_panel(cfg)
    SYNTHETIC = False
except Exception as error:
    print("no real data yet, using the synthetic stand-in:")
    print(" ", str(error).splitlines()[0])
    from src.synthetic import synthetic_panel
    panel = synthetic_panel(cfg)
    SYNTHETIC = True

clean = preprocess(panel, cfg).panel[list(panel.columns)]
clean.shape

## Building the design matrix

One function, shared by training, evaluation and simulation.

In [ ]:
from src.features import build_features

X, y, spec = build_features(clean, cfg)
print(f'X = {X.shape}  (samples, timesteps, features)')
print(f'target: {spec.target_name}')
print(f'dropped states: {spec.dropped_states or "none"}')

## Where each column came from

Provenance is recorded, so nothing downstream has to parse a column name.

In [ ]:
import pandas as pd

pd.DataFrame([
    {'column': c,
     'raw_variable': spec.origins[c].raw_variable,
     'transform': spec.origins[c].transform,
     'lag': spec.origins[c].lag}
    for c in spec.columns
]).head(20)

## The leakage check

Overwrite everything after a cut date and every earlier sample must be unchanged. This catches any path future data could take, including through rolling windows and the spatial graph.

In [ ]:
import numpy as np

cut = pd.Timestamp(cfg.data.start_date) + pd.DateOffset(years=6)

tampered = clean.copy()
tampered.loc[(slice(None), slice(cut, None)), :] = 1e6
tampered_X, _, _ = build_features(tampered, cfg)

past = spec.sample_index.get_level_values('date') < cut
print(f'samples before {cut.date()}: {past.sum()} of {len(past)}')
print('identical:', np.allclose(X[past], tampered_X[past]))

## Dimensionality

brain.md Q-10: lag features and the sequence window encode the same history twice. The ratio of samples to flattened inputs is the thing to watch.

In [ ]:
from src.features import flatten

flat = flatten(X)
print(f'{X.shape[0]} samples / {flat.shape[1]} flat inputs = {X.shape[0] / flat.shape[1]:.2f}')